In [13]:
import geopandas as gpd
import networkx as nx
from centerline.geometry import Centerline
from shapely.ops import polygonize
import pandas as pd

In [14]:
crs = 27700

In [15]:
def create_blocks_from_roads(roads_and_buffers):
    """
    Create polygons representing spaces between roads.
    This is required instead of simply creating the adjacency graph from non-buffered roads
    because the roads will be buffered on the map and by buffering some polygons
    disappear. We don't want these disappearing polygons to have an effect on the adjacency graph

    Parameters:
    - gdf_roads: GeoDataFrame with LineString geometries of roads
    - boundary: boundary of the city
    - buffer_distance: distance to buffer roads (in CRS units)

    Returns:
    - GeoDataFrame of polygons representing the spaces between roads
    """

    gdfs = []
    for road_gdf, buffer_size in roads_and_buffers:
        gdfs.append(road_gdf.buffer(buffer_size))

    # Merge buffered roads into a single polygon
    roads_union_gdf = gpd.GeoDataFrame(geometry=pd.concat(gdfs), crs=crs).dissolve().explode()

    return roads_union_gdf

In [4]:
def unbuffer_roads(roads_gdf, boundary_gdf, interpolation_distance=10):
    # Combine the roads and the boundary
    buffered_boundary = boundary_gdf.boundary.buffer(5) # less than 5 doesn't work well
    geometry = pd.concat([roads_gdf.geometry, buffered_boundary])
    road_line_gdf = gpd.GeoDataFrame(geometry=geometry, crs=crs).dissolve().explode()

    # Unbuffer the road polygon to get a multilinestring. This is the time-consuming step
    road_line_gdf['geometry'] = road_line_gdf.geometry.apply(
        lambda geom: Centerline(geom, interpolation_distance=interpolation_distance).geometry
    )

    # Join the multilinestring in case unbuffering caused small gaps
    road_line_gdf = road_line_gdf.dissolve().explode()

    # Get the polygons between the lines and convert to gdf
    polygons = list(polygonize(road_line_gdf.geometry))
    polygons_gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs)

    return polygons_gdf

In [5]:
def get_adjacency_graph(polygon_gdf):

    # Perform a spatial join using 'touches'
    touches_pairs = gpd.sjoin(polygon_gdf, polygon_gdf, how="inner", predicate="touches")

    # Reset index
    touches_pairs = touches_pairs.reset_index(names='index_left')

    # Remove self-joins and keep only one direction (avoid duplicate edges)
    touches_pairs = touches_pairs[touches_pairs['index_left'] < touches_pairs['index_right']]

    # Join to get both polygon geometries
    touches_pairs = touches_pairs.merge(polygon_gdf, left_on='index_right', right_index=True, suffixes=['_left', '_right'])

    # Only keep intersections that are not points - time-consuming step
    touches_pairs['intersection'] = touches_pairs.apply(lambda row: row['geometry_left'].intersection(row['geometry_right']), axis=1)
    touches_pairs = touches_pairs[touches_pairs['intersection'].geom_type != 'Point']

    # Calculate intersection length for filtering later
    touches_pairs['length'] = touches_pairs['intersection'].length

    # Create the graph
    G = nx.from_pandas_edgelist(
        touches_pairs,
        source='index_left',
        target='index_right',
        edge_attr='length'
    )

    # Calculate node area for equitable coloring later
    area_dict = polygon_gdf.area.to_dict()
    nx.set_node_attributes(G, area_dict, 'area')

    return G

In [16]:
greater_london_gdf = gpd.read_file('data/raw/gla/London_GLA_Boundary.shp')
roads_gdf = gpd.read_file('data/intermediary/roads/roads.shp')
major_roads_gdf = roads_gdf[roads_gdf.fclass.isin(['motorway', 'trunk', 'primary'])]
middle_roads_gdf = roads_gdf[roads_gdf.fclass.isin(['secondary', 'tertiary'])]
minor_roads_gdf = roads_gdf[roads_gdf.fclass.isin(['unclassified', 'residential', 'living_street', 'pedestrian', 'busway'])]

In [20]:
# Buffer the roads and output to file
buffered_roads = create_blocks_from_roads([(major_roads_gdf, 5)])
buffered_roads.to_file('data/intermediary/buffered_roads')

buffered_roads_all = create_blocks_from_roads([
    (major_roads_gdf, 5),
    (middle_roads_gdf, 2),
    (minor_roads_gdf, 1)
])
buffered_roads_all.to_file('data/intermediary/buffered_roads_all')

In [16]:
# Unbuffer the roads to a multilinestring and output the polygons to file
buffered_roads = gpd.read_file('data/intermediary/buffered_roads/buffered_roads.shp')
polygons = unbuffer_roads(buffered_roads, greater_london_gdf)
polygons.to_file('data/intermediary/polygons')


In [17]:
# Create an adjacency graph from the polygons
polygons = gpd.read_file('data/intermediary/polygons/buffered_polygons.shp')
graph = get_adjacency_graph(polygons)
nx.write_graphml(graph, 'data/intermediary/graph.graphml')